In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

In [2]:
# Load embeddings
embeddings = np.load("data/embedded/news_embeddings_final.npy")
news_ids   = np.load("data/embedded/news_ids.npy", allow_pickle=True)

# Load behavior data
# behaviors_clean.csv includes hour_sin, hour_cos, day_sin, day_cos columns
# produced by Data_Processing.ipynb (Cell 17 saves the augmented version)
behaviors = pd.read_csv("data/cleaned/behaviors_clean.csv")

print("Embeddings shape:", embeddings.shape)
print("Number of news IDs:", len(news_ids))
print("Behaviors shape:", behaviors.shape)

behaviors.head()

Embeddings shape: (51282, 768)
Number of news IDs: 51282
Behaviors shape: (5723002, 5)


,user_id,history,news_id,label,category
0,U13740,"['N55189', 'N42782', 'N34694', 'N45794', 'N184...",N55689,1,sports
1,U13740,"['N55189', 'N42782', 'N34694', 'N45794', 'N184...",N35729,0,news
2,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N20678,0,sports
3,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N39317,0,news
4,U91836,"['N31739', 'N6072', 'N63045', 'N23979', 'N3565...",N58114,0,autos


In [3]:
news_vector_map = {
    news_id: embeddings[i]
    for i, news_id in enumerate(news_ids)
}

embedding_dim = embeddings.shape[1]   # 768
TIME_DIM      = 4                      # hour_sin, hour_cos, day_sin, day_cos
STATE_DIM     = embedding_dim + TIME_DIM  # 772

print("Embedding dim:", embedding_dim)
print("State dim (embedding + time):", STATE_DIM)

Embedding dim: 768
State dim (embedding + time): 772


In [4]:
sample_id = news_ids[0]
print("Sample news_id:", sample_id)
print("Vector[:5]:", news_vector_map[sample_id][:5])

Sample news_id: N55528
Vector[:5]: [-0.01886897  0.06061954  0.03854043  0.00348515  0.04707707]


In [5]:
grouped = behaviors.groupby(["user_id", "history"])
print("Total groups:", len(grouped))

Total groups: 49108


## Updated `get_state` — recency-weighted history + temporal context

Uses exponential decay (`exp(-λ·i)`) on the reversed history so the most recent article has highest weight. Appends 4 cyclical time features (hour_sin/cos, day_sin/cos) so the bandit can learn time-of-day and day-of-week patterns.

In [6]:
def get_state(history, time_features=None, decay_lambda=0.5):
    """
    Parameters
    ----------
    history       : list of news_id strings (most-recent last)
    time_features : np.array of shape (4,) = [hour_sin, hour_cos, day_sin, day_cos]
                    Pass None to get a pure embedding state (embedding_dim dims).
    decay_lambda  : controls recency bias. 0 = uniform mean, higher = focus on recent.
    
    Returns
    -------
    state vector of shape (STATE_DIM,) = (772,) when time_features provided,
    or (embedding_dim,) = (768,) when time_features=None.
    """
    try:
        history = eval(history) if isinstance(history, str) else history
    except:
        history = []

    vectors = []
    weights = []

    for i, n in enumerate(history[::-1]):   # reverse → index 0 = most recent
        if n in news_vector_map:
            vectors.append(news_vector_map[n])
            weights.append(np.exp(-decay_lambda * i))  # exponential decay

    if len(vectors) == 0:
        base = np.zeros(embedding_dim)
    else:
        vectors = np.array(vectors)
        weights = np.array(weights)
        base = np.average(vectors, axis=0, weights=weights)

    if time_features is not None:
        return np.concatenate([base, time_features])
    return base

In [7]:
def extract_candidates(group, news_vector_map):
    news_ids_col = group["news_id"].values
    labels       = group["label"].values

    candidates   = []
    valid_labels = []

    for nid, lbl in zip(news_ids_col, labels):
        if nid in news_vector_map:
            candidates.append(news_vector_map[nid])
            valid_labels.append(lbl)

    if len(candidates) == 0:
        return None, None

    return candidates, np.array(valid_labels)

In [8]:
TIME_COLS = ["hour_sin", "hour_cos", "day_sin", "day_cos"]

def get_time_features(group):
    """Extract the 4 temporal features from the first row of a group."""
    row = group.iloc[0]
    if all(c in row.index for c in TIME_COLS):
        return row[TIME_COLS].values.astype(float)
    return None   # graceful fallback if columns are missing

## Metric helpers

In [9]:
def compute_auc(labels, scores):
    if len(set(labels)) < 2:
        return None
    return roc_auc_score(labels, scores)

def compute_mrr(labels, scores):
    order = np.argsort(scores)[::-1]
    for i, rel in enumerate(labels[order]):
        if rel == 1:
            return 1 / (i + 1)
    return 0

def compute_ndcg_at_k(labels, scores, k):
    order        = np.argsort(scores)[::-1]
    sorted_labels = labels[order][:k]
    dcg   = sum(rel / np.log2(i + 2) for i, rel in enumerate(sorted_labels))
    ideal = sorted(labels, reverse=True)[:k]
    idcg  = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0

## Train / test split

Split groups **by time** so the bandit is always evaluated on future impressions it hasn't trained on. This gives honest, unbiased metric estimates.

In [10]:
# Sort behaviors by time before grouping so the split is chronological
if "time" in behaviors.columns:
    behaviors_sorted = behaviors.sort_values("time").reset_index(drop=True)
else:
    behaviors_sorted = behaviors.copy()

grouped_sorted = behaviors_sorted.groupby(["user_id", "history"], sort=False)
grouped_list   = list(grouped_sorted)

split_idx    = int(len(grouped_list) * 0.8)
train_groups = grouped_list[:split_idx]
test_groups  = grouped_list[split_idx:]

print(f"Total groups : {len(grouped_list)}")
print(f"Train groups : {len(train_groups)}  ({split_idx / len(grouped_list):.0%})")
print(f"Test  groups : {len(test_groups)}  ({len(test_groups) / len(grouped_list):.0%})")

Total groups : 49108
Train groups : 39286  (80%)
Test  groups : 9822  (20%)


In [11]:
# Pre-compute states for every group (avoids recomputing inside the hot loop)
state_cache = {}

for (user_id, history), group in grouped_list:
    tf = get_time_features(group)
    state_cache[(user_id, history)] = get_state(history, tf)

print("State cache built.  Sample shape:", next(iter(state_cache.values())).shape)

State cache built.  Sample shape: (768,)


## Bandit 1 — ε-Greedy Contextual Bandit

In [12]:
class ContextualBandit:
    """Linear epsilon-greedy bandit.  Works with any state dimensionality."""

    def __init__(self, dim, epsilon=0.1):
        self.dim     = dim
        self.epsilon = epsilon
        self.theta   = np.zeros(dim)

    def predict(self, state, action):
        return np.dot(self.theta, state * action)

    def select_action(self, state, candidates):
        if np.random.rand() < self.epsilon:
            return np.random.randint(len(candidates))
        scores = [self.predict(state, a) for a in candidates]
        return np.argmax(scores)

    def update(self, state, action, reward, lr=0.01):
        self.theta += lr * reward * (state * action)

    def get_scores(self, state, candidates):
        return np.array([self.predict(state, a) for a in candidates])

## Generic training + evaluation loop

Single function used by all three bandit types.  `train_data` updates the bandit; `eval_data` only collects metrics (no update) — this is the key change for unbiased evaluation.

In [13]:
def run_bandit(bandit, train_data, eval_data=None, max_steps=None, max_candidates=20):
    """
    Parameters
    ----------
    bandit          : any bandit with .get_scores(), .select_action(), .update()
    train_data      : list of ((user_id, history), group) tuples — bandit updates here
    eval_data       : list of ((user_id, history), group) tuples — metrics only, no update.
                      If None, metrics are collected during training (online eval).
    max_steps       : cap on number of training steps
    max_candidates  : limit candidates per impression for speed
    
    Returns
    -------
    dict with CTR, AUC, MRR, nDCG@5, nDCG@10
    """

    # ── Training phase ────────────────────────────────────────────────────────
    total_reward, total_steps = 0, 0
    train_auc, train_mrr, train_ndcg5, train_ndcg10 = [], [], [], []

    for i, ((user_id, history), group) in enumerate(tqdm(train_data, desc="Train")):

        if max_steps and i >= max_steps:
            break

        state      = state_cache.get((user_id, history), get_state(history, get_time_features(group)))
        candidates, labels = extract_candidates(group, news_vector_map)

        if candidates is None:
            continue

        if len(candidates) > max_candidates:
            idx        = np.random.choice(len(candidates), max_candidates, replace=False)
            candidates = [candidates[j] for j in idx]
            labels     = labels[idx]

        scores = bandit.get_scores(state, candidates)

        auc = compute_auc(labels, scores)
        if auc is not None:
            train_auc.append(auc)
        train_mrr.append(compute_mrr(labels, scores))
        train_ndcg5.append(compute_ndcg_at_k(labels, scores, 5))
        train_ndcg10.append(compute_ndcg_at_k(labels, scores, 10))

        action_idx = bandit.select_action(state, candidates)
        reward     = labels[action_idx]
        bandit.update(state, candidates[action_idx], reward)

        total_reward += reward
        total_steps  += 1

    train_results = {
        "train_CTR"    : total_reward / max(total_steps, 1),
        "train_AUC"    : np.mean(train_auc)   if train_auc   else float("nan"),
        "train_MRR"    : np.mean(train_mrr)   if train_mrr   else float("nan"),
        "train_nDCG@5" : np.mean(train_ndcg5) if train_ndcg5 else float("nan"),
        "train_nDCG@10": np.mean(train_ndcg10) if train_ndcg10 else float("nan"),
    }

    # ── Evaluation phase (held-out test set) ──────────────────────────────────
    if eval_data is None:
        return train_results   # online eval only

    eval_auc, eval_mrr, eval_ndcg5, eval_ndcg10 = [], [], [], []
    eval_reward, eval_steps = 0, 0

    for (user_id, history), group in tqdm(eval_data, desc="Eval"):

        state      = state_cache.get((user_id, history), get_state(history, get_time_features(group)))
        candidates, labels = extract_candidates(group, news_vector_map)

        if candidates is None:
            continue

        if len(candidates) > max_candidates:
            idx        = np.random.choice(len(candidates), max_candidates, replace=False)
            candidates = [candidates[j] for j in idx]
            labels     = labels[idx]

        scores = bandit.get_scores(state, candidates)   # NO update in eval

        auc = compute_auc(labels, scores)
        if auc is not None:
            eval_auc.append(auc)
        eval_mrr.append(compute_mrr(labels, scores))
        eval_ndcg5.append(compute_ndcg_at_k(labels, scores, 5))
        eval_ndcg10.append(compute_ndcg_at_k(labels, scores, 10))

        # CTR: pick greedily (no exploration during eval)
        action_idx   = np.argmax(scores)
        eval_reward += labels[action_idx]
        eval_steps  += 1

    eval_results = {
        "test_CTR"    : eval_reward / max(eval_steps, 1),
        "test_AUC"    : np.mean(eval_auc)   if eval_auc   else float("nan"),
        "test_MRR"    : np.mean(eval_mrr)   if eval_mrr   else float("nan"),
        "test_nDCG@5" : np.mean(eval_ndcg5) if eval_ndcg5 else float("nan"),
        "test_nDCG@10": np.mean(eval_ndcg10) if eval_ndcg10 else float("nan"),
    }

    return {**train_results, **eval_results}

### Run ε-Greedy bandit (train → held-out eval)

In [14]:
bandit_eps = ContextualBandit(dim=STATE_DIM, epsilon=0.1)
results_eps = run_bandit(bandit_eps, train_groups, eval_data=test_groups, max_steps=5000)

print("\n── ε-Greedy Results ──")
for k, v in results_eps.items():
    print(f"  {k:<18}: {v:.4f}")

Train:   0%|          | 0/39286 [00:00<?, ?it/s]


ValueError: shapes (772,) and (768,) not aligned: 772 (dim 0) != 768 (dim 0)

## Bandit 2 — LinUCB

UCB = θᵀx + α·√(xᵀA⁻¹x) 
Uses vectorised batch scoring and the Sherman-Morrison rank-1 update to avoid recomputing `inv(A)` from scratch each step.

In [ ]:
class LinUCB:
    """Disjoint LinUCB.  Maintains a single shared A / b across all arms."""

    def __init__(self, dim, alpha=1.0):
        self.dim   = dim
        self.alpha = alpha
        self.A     = np.identity(dim)
        self.A_inv = np.identity(dim)   # maintained via Sherman-Morrison
        self.b     = np.zeros(dim)

    def get_theta(self):
        return self.A_inv @ self.b

    def select_action(self, state, candidates):
        theta = self.get_theta()
        X     = np.array([state * a for a in candidates])   # (n, dim)

        mean        = X @ theta
        temp        = X @ self.A_inv
        uncertainty = np.sqrt(np.clip(np.sum(temp * X, axis=1), 0, None))
        ucb_scores  = mean + self.alpha * uncertainty

        return np.argmax(ucb_scores)

    def get_scores(self, state, candidates):
        theta = self.get_theta()
        X     = np.array([state * a for a in candidates])
        return X @ theta

    def update(self, state, action, reward):
        x         = state * action
        self.A   += np.outer(x, x)
        self.b   += reward * x

        # Sherman-Morrison: O(d²) instead of O(d³)
        A_inv_x   = self.A_inv @ x
        denom     = 1 + x @ A_inv_x
        self.A_inv -= np.outer(A_inv_x, A_inv_x) / denom

### Run LinUCB (train → held-out eval)

In [ ]:
bandit_ucb = LinUCB(dim=STATE_DIM, alpha=1.0)
results_ucb = run_bandit(bandit_ucb, train_groups, eval_data=test_groups, max_steps=5000)

print("\n── LinUCB Results ──")
for k, v in results_ucb.items():
    print(f"  {k:<18}: {v:.4f}")

## Bandit 3 — Linear Thompson Sampling

Samples a θ from the posterior approximation to encourage exploration probabilistically rather than via an explicit UCB bonus.

In [ ]:
class LinTS:
    """Linear Thompson Sampling with fast diagonal-noise posterior approximation."""

    def __init__(self, dim, v=1.0):
        self.dim   = dim
        self.v     = v
        self.A     = np.identity(dim)
        self.A_inv = np.identity(dim)
        self.b     = np.zeros(dim)

    def get_theta(self):
        return self.A_inv @ self.b

    def sample_theta(self):
        mean = self.get_theta()
        return mean + self.v * np.random.randn(self.dim)   # fast diagonal approx

    def select_action(self, state, candidates):
        theta_sample = self.sample_theta()
        scores = [np.dot(theta_sample, state * a) for a in candidates]
        return np.argmax(scores)

    def get_scores(self, state, candidates):
        theta = self.get_theta()
        return np.array([np.dot(theta, state * a) for a in candidates])

    def update(self, state, action, reward):
        x         = state * action
        self.A   += np.outer(x, x)
        self.b   += reward * x

        # Sherman-Morrison
        A_inv_x   = self.A_inv @ x
        denom     = 1 + x @ A_inv_x
        self.A_inv -= np.outer(A_inv_x, A_inv_x) / denom

### Run LinTS (train → held-out eval)

In [ ]:
bandit_ts = LinTS(dim=STATE_DIM, v=1.0)
results_ts = run_bandit(bandit_ts, train_groups, eval_data=test_groups, max_steps=5000)

print("\n── LinTS Results ──")
for k, v in results_ts.items():
    print(f"  {k:<18}: {v:.4f}")

## Comparison — all three bandits

In [ ]:
summary = pd.DataFrame([
    {"bandit": "ε-Greedy",  **results_eps},
    {"bandit": "LinUCB",    **results_ucb},
    {"bandit": "LinTS",     **results_ts},
]).set_index("bandit")

print(summary.to_string())
summary

## Hyperparameter sweeps

All sweeps use the same 80/20 train/test split and are capped at 5,000 training steps for speed.

### ε-Greedy — sweep epsilon

In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.2]
results_eps_sweep = []

for eps in epsilons:
    print(f"  epsilon = {eps}")
    b   = ContextualBandit(dim=STATE_DIM, epsilon=eps)
    res = run_bandit(b, train_groups, eval_data=test_groups, max_steps=5000)
    res["epsilon"] = eps
    results_eps_sweep.append(res)

pd.DataFrame(results_eps_sweep).set_index("epsilon")

### LinUCB — sweep alpha

In [ ]:
alphas = [0.1, 0.5, 1.0, 2.0]
results_ucb_sweep = []

for alpha in alphas:
    print(f"  alpha = {alpha}")
    b   = LinUCB(dim=STATE_DIM, alpha=alpha)
    res = run_bandit(b, train_groups, eval_data=test_groups, max_steps=5000)
    res["alpha"] = alpha
    results_ucb_sweep.append(res)

pd.DataFrame(results_ucb_sweep).set_index("alpha")

### LinTS — sweep v

In [ ]:
vs = [0.1, 0.5, 1.0, 2.0]
results_ts_sweep = []

for v in vs:
    print(f"  v = {v}")
    b   = LinTS(dim=STATE_DIM, v=v)
    res = run_bandit(b, train_groups, eval_data=test_groups, max_steps=5000)
    res["v"] = v
    results_ts_sweep.append(res)

pd.DataFrame(results_ts_sweep).set_index("v")

### Decay lambda sweep (LinUCB, alpha=1.0)

Tune the exponential decay rate λ used in `get_state()`.  λ=0 → uniform mean, larger λ → focus heavily on the most recent article.

In [ ]:
decay_lambdas = [0.0, 0.1, 0.5, 1.0, 2.0]
results_decay = []

def get_state_lambda(history, time_features=None, lam=0.5):
    try:
        history = eval(history) if isinstance(history, str) else history
    except:
        history = []
    vectors, weights = [], []
    for i, n in enumerate(history[::-1]):
        if n in news_vector_map:
            vectors.append(news_vector_map[n])
            weights.append(np.exp(-lam * i) if lam > 0 else 1.0)
    if not vectors:
        base = np.zeros(embedding_dim)
    else:
        base = np.average(np.array(vectors), axis=0, weights=np.array(weights))
    if time_features is not None:
        return np.concatenate([base, time_features])
    return base

for lam in decay_lambdas:
    print(f"  lambda = {lam}")

    # Rebuild cache for this lambda
    lam_cache = {}
    for (user_id, history), group in grouped_list:
        tf = get_time_features(group)
        lam_cache[(user_id, history)] = get_state_lambda(history, tf, lam=lam)

    # Temporarily replace state_cache
    original_cache = state_cache.copy()
    state_cache.update(lam_cache)

    b   = LinUCB(dim=STATE_DIM, alpha=1.0)
    res = run_bandit(b, train_groups, eval_data=test_groups, max_steps=5000)
    res["lambda"] = lam
    results_decay.append(res)

    state_cache.clear()
    state_cache.update(original_cache)

pd.DataFrame(results_decay).set_index("lambda")